# Les files — Exercice 2 : `passe_ton_tour`, `nb_occurences`, `last_is_first` (correction)

Ce notebook contient, pour chaque question, trois blocs **toujours séparés** :

| Bloc | Contenu |
|:---|:---|
| 📝 **ÉNONCÉ** | la question, recopiée du sujet |
| ✅ **RÉSOLUTION** | la réponse attendue (code et/ou texte) |
| 💡 **EXPLICATION PÉDAGOGIQUE** | pourquoi ça marche, le lien avec le cours, les erreurs à éviter. **N'est pas à recopier.** |

**Notions travaillées :** utiliser une file **uniquement par ses méthodes**, parcourir une file sans la modifier (file auxiliaire).

Exécuter les cellules **dans l'ordre** (*Cell → Run All*).

---
# MISE EN PLACE

> Dans la suite, on utilisera exclusivement uniquement des objets de la classe File définie dans l'exercice 1.
> (Interdit d'utiliser le type list de python.)

On importe la classe `File` de l'exercice 1 : le fichier `file.py` doit être **dans le même dossier** que ce notebook.

**Aucune liste python, aucune boucle `for`** : on ne manipule les files qu'avec `enfile`, `defile`, `est_vide` (et `taille`). Pour les vérifications, on fabrique les files par des `enfile` successifs, et on compare leur **affichage** au texte attendu.

Une file s'affiche sous la forme `File(['a', 'b', 'c'])` : la **tête** est le premier élément (à gauche, c'est lui qu'on défile), la **queue** est le dernier (à droite, c'est là qu'on enfile).

In [1]:
from file import File

f = File()
f.enfile("a")
f.enfile("b")
f.enfile("c")
print(f)

File(['a', 'b', 'c'])


---
# QUESTION 1

#### 📝 ÉNONCÉ

> 1. Écrire une fonction `passe_ton_tour(f)` qui place en queue de la file `f` l'élément en tête.

#### ✅ RÉSOLUTION

In [2]:
def passe_ton_tour(f):
    if not f.est_vide():
        f.enfile(f.defile())

*Vérification (non demandée) :*

In [3]:
f = File()
f.enfile(1)
f.enfile(2)
f.enfile(3)
f.enfile(4)
passe_ton_tour(f)
print(f)
assert str(f) == "File([2, 3, 4, 1])"

g = File()
passe_ton_tour(g)        # file vide : pas d'erreur
assert g.est_vide()

File([2, 3, 4, 1])


#### 💡 EXPLICATION PÉDAGOGIQUE

- `defile()` retire la tête et la **renvoie** ; `enfile(...)` l'ajoute en queue. On peut donc tout écrire en une ligne : `f.enfile(f.defile())`.
- **Cas de la file vide :** `defile` déclencherait une erreur. On choisit de ne rien faire.
- La fonction **ne renvoie rien** : elle **modifie** la file reçue. Une file est un objet : la fonction reçoit le **même objet** que l'appelant, pas une copie.

---
# QUESTION 2

#### 📝 ÉNONCÉ

> 2. Écrire une fonction `nb_occurences(f, val)` qui renvoie le nombre d'éléments de la file `f` qui sont égaux à `val`. On fera en de sorte que, après l'appel de la fonction, la file passée en argument soit identique à ce qu'elle était initialement.

#### ✅ RÉSOLUTION

In [4]:
def nb_occurences(f, val):
    aux = File()
    compteur = 0
    while not f.est_vide():
        el = f.defile()
        if el == val:
            compteur = compteur + 1
        aux.enfile(el)
    while not aux.est_vide():
        f.enfile(aux.defile())
    return compteur

*Vérification (non demandée) :*

In [5]:
f = File()
f.enfile(3)
f.enfile(1)
f.enfile(3)
f.enfile(2)
f.enfile(3)
print(nb_occurences(f, 3), "fois le nombre 3")
print("La file après l'appel :", f)

assert nb_occurences(f, 3) == 3
assert nb_occurences(f, 7) == 0
assert str(f) == "File([3, 1, 3, 2, 3])"     # file inchangée
assert nb_occurences(File(), 1) == 0

3 fois le nombre 3
La file après l'appel : File([3, 1, 3, 2, 3])


#### 💡 EXPLICATION PÉDAGOGIQUE

- **La technique clé de tout le TD : la file auxiliaire.** Avec une file, on ne voit que la **tête**. Pour regarder tous les éléments, on **vide** la file dans une file auxiliaire `aux`, en traitant chaque élément au passage, puis on **reverse** `aux` dans la file de départ. Comme une file est FIFO, les éléments ressortent de `aux` **dans le même ordre** : la file est **identique** à ce qu'elle était au départ.
  ```python
  aux = File()
  while not f.est_vide():
      el = f.defile()
      ...                    # traiter el
      aux.enfile(el)
  while not aux.est_vide():
      f.enfile(aux.defile())
  ```
- **Pourquoi la première boucle s'arrête ?** Parce qu'on enfile dans `aux`, et **pas** dans `f` : `f` perd un élément à chaque tour et finit vide. ⚠️ Si on ré-enfilait `el` dans `f`, elle ne se viderait jamais et la boucle `while not f.est_vide()` ne s'arrêterait **jamais**.
- **Pourquoi pas `for i in range(f.taille())` ?** Ce serait parcourir la file « case par case », comme un tableau. Une file ne s'utilise pas ainsi : on défile **tant qu'elle n'est pas vide**, comme une vraie file d'attente.
- **Coût :** chaque élément est défilé et enfilé deux fois (aller dans `aux`, retour dans `f`) : de l'ordre de `2n` opérations, donc un coût **linéaire** (proportionnel à `n`). En mémoire, presque rien de plus : les éléments ne sont pas **copiés**, ils **passent** d'une file à l'autre ; à tout moment, `f` et `aux` contiennent ensemble les `n` éléments.
- ⚠️ **Erreur fréquente :** oublier la deuxième boucle. Le résultat est juste, mais `f` est **vide** à la fin, ce que l'énoncé interdit.

- Ici, à chaque élément défilé, on augmente le compteur s'il est égal à `val`, puis on le range dans `aux`.

| Étape | `f` | `aux` | `compteur` |
|:---|:---|:---|:---|
| départ | `3 1 3 2 3` | | 0 |
| après la 1re boucle | | `3 1 3 2 3` | 3 |
| après la 2e boucle | `3 1 3 2 3` | | 3 |

- L'énoncé écrit *occurences* (et « en de sorte ») ; l'orthographe correcte est *occu**rr**ences*, mais on garde le nom de fonction demandé.

---
# QUESTION 3

#### 📝 ÉNONCÉ

> 3. Écrire une fonction `last_is_first(f)` qui place le dernier élément de la file `f` en premier.

#### ✅ RÉSOLUTION

In [6]:
def last_is_first(f):
    if f.est_vide():
        return
    aux = File()
    while not f.est_vide():
        el = f.defile()
        if f.est_vide():
            dernier = el
        else:
            aux.enfile(el)
    f.enfile(dernier)
    while not aux.est_vide():
        f.enfile(aux.defile())

*Vérification (non demandée) :*

In [7]:
f = File()
f.enfile("a")
f.enfile("b")
f.enfile("c")
f.enfile("d")
last_is_first(f)
print(f)
assert str(f) == "File(['d', 'a', 'b', 'c'])"

g = File()
last_is_first(g)                 # file vide
assert g.est_vide()

g.enfile(1)
last_is_first(g)                 # un seul élément
assert str(g) == "File([1])"

g.enfile(2)
last_is_first(g)                 # deux éléments
assert str(g) == "File([2, 1])"

File(['d', 'a', 'b', 'c'])


#### 💡 EXPLICATION PÉDAGOGIQUE

- **Comment reconnaître le dernier élément ?** On ne connaît que la tête… mais juste après avoir défilé un élément, si la file est **devenue vide**, c'est que cet élément était le **dernier**. On le met de côté dans la variable `dernier` ; tous les autres vont dans `aux`.
- Ensuite, on enfile `dernier` **en premier** dans `f` (vide à ce moment-là), puis on reverse `aux` derrière lui : les autres gardent leur ordre.

| Étape | `f` | `aux` | `dernier` |
|:---|:---|:---|:---|
| départ | `a b c d` | | |
| après la boucle | | `a b c` | `d` |
| `f.enfile(dernier)` | `d` | `a b c` | `d` |
| après le reversement | `d a b c` ✔ | | |

- **Cas de la file vide :** il n'y a pas de dernier élément (et `dernier` ne serait jamais défini) : on sort tout de suite avec `return`.
- ⚠️ **Erreur fréquente :** reverser `aux` **avant** d'enfiler `dernier` : on obtiendrait `a b c d`, la file de départ.